In [1]:
from syto.data.pseudobulk_hdf5_utils import PseudobulkHDF5Reader
import pandas as pd
from typing import Dict, Any
from copy import deepcopy
import pickle
from syto.data.atlases.uxm_atlases import UXMMethylationAtlas
from baselines.deconvolution.uxm.uxm import prepare_reads_for_uxm, mark_records_methyl_state
from syto.data import LOYFER_CELL_TYPE_MATCH_DICT
import json

In [2]:
pseudobulk_path = "/home/luna.kuleuven.be/u0169940/Repos/syto/tmp/pseudobulk_generation/pseudobulk.h5"

In [3]:
reader = PseudobulkHDF5Reader(pseudobulk_path)

In [4]:
iterator = reader.iter_pseudobulk_read_subsets("train", class_label_column="original_label")

In [5]:
x = iterator.__next__()

In [6]:
def _build_uxm_input(
        input: pd.DataFrame, uxm_atlas: pd.DataFrame, ref_cells: list
    ) -> Dict[str, Any]:
        """
        Build UXM-compatible input from prepared reads.

        Computes per-region scaling factors and methylation counts
        in the format expected by ``decon_single_samp``.
        """
        input

        results_agg = (
            input[input["NCPGS"] > 3]
            .groupby("name")
            .aggregate({"record_M": "sum", "record_U": "sum", "record_X": "sum"})
            .reset_index()
        )
        results_agg["count"] = (
            results_agg["record_M"] + results_agg["record_U"] + results_agg["record_X"]
        )
        results_agg["sf"] = results_agg["record_U"] / results_agg["count"]
        results_agg["direction"] = "U"
        sample_name = "sample"

        sf = deepcopy(results_agg[["name", "direction"]])
        sf[sample_name] = results_agg["sf"]
        counts = results_agg[["name", "direction", "count"]]
        counts.columns = ["name", "direction", sample_name]

        return {
            "scaling_factors": sf,
            "counts": counts,
        }

In [7]:
atlas_path = "/home/luna.kuleuven.be/u0169940/Repos/UXM_deconv/supplemental/Atlas.U25.l4.hg38.full.tsv"
UXMatlas = UXMMethylationAtlas(atlas_name="U25l4", 
                                    reference_genome="hg38" if "hg38" in atlas_path else "hg19",
                                    atlas_path=atlas_path,
                                    sep="\t")

In [8]:
labels_dict_path = "/home/luna.kuleuven.be/u0169940/Repos/syto/App/labels_dict.json"

In [9]:
with open(labels_dict_path, "r", encoding="utf-8") as f:
            # JSON keys are strings; convert to {int: str}
            labels_dict = json.load(f)

In [11]:
z = mark_records_methyl_state(x[0], methyl_tr=0.75, unmethyl_tr=0.25)

In [19]:
uxm_input = _build_uxm_input(z, UXMatlas._atlas, ref_cells=[x for x in list(UXMatlas._atlas["target"].unique()) if x not in "Megakaryocytes"])